# Experiment 09 — K-ratio features and future vault performance

Based on [experiment 05's causal feature diagnostics](05-opt-score-calibration-selection-cash.ipynb), the [reviewed implementation plan](09-k-ratio-feature-forecast-plan.md), and the [post-review amendment](09-k-ratio-feature-forecast-amendment.md). This notebook is corrected and rerun in place; the original run is archived.

## Goal

Measure the prediction power of **individual historical vault features**, adding **K-ratio** to the existing CAGR, Sharpe, Sortino, maximum drawdown and Calmar inputs. Predict future **CAGR, Sharpe and K-ratio** over **7, 30 and 90 days**, using four-day forecast origins. Support **missing=True** so a missing long lookback does not exclude a vault with usable shorter inputs. This is feature forecasting, not composite scoring or portfolio weighting.

## Method

Use pinned repaired data, daily share NAV with causal forward filling and strictly matured future labels. Add six K inputs: **30, 60, 90, 180 and 360 days (one year), plus max (all recorded history)**. Max starts at each vault's first actual positive daily NAV and ends at T−1; it carries the latest known NAV over missing calendar days, requires at least three fitted points, and never compresses gaps or backfills before the first observation. Its duration varies by vault/date and is not certified inception. Forecasts use only history through T−1, with no additional gap; future outcomes never extend a feature window. The loader includes the earliest actual archive date, even if it precedes the usual 731-day buffer. Missing=True preserves shorter-window forecasts when year/max is unavailable.

Use exact lookback log returns for CAGR inputs, robust training-only asinh/standardisation with training-range/±5z input bounds and training 0.5%–99.5% prediction bounds. Sharpe uses scaled pseudo-Huber loss; K uses asinh targets. Compare fixed monthly forecasts with and without K on identical vault/date rows, with shared availability controls, constant/zero/persistence baselines, compact and path-decomposition companions. Retain four-window K ablations, adding year only, max only, and both with the same controls. Model preprocessing is training-only. K uses ordinary intercept OLS on all L+1 daily wealth points, including the anchor, and sqrt(365)/(L+1); undefined ratios stay missing. Report 2026 by default, with earlier-history diagnostics separately. These dates were previously researched; results are not an untouched holdout.

## Key new insights

All historical metrics and realised targets now use causal daily forward fill; the unchanged archive and pre-fill results are preserved in `09-before-sparse-fill`. Adding six K inputs (30/60/90/180/360 days and max recorded history) improves primary mean rank IC in **6 of 6** supported target/horizon comparisons. This is a measured result of fixed amended models on previously researched history, not a general claim about K or an untouched holdout.

- **7-day CAGR:** old/new rank IC 0.018/0.021; delta +0.003; modelling-scale MAE change +0.5%; 45 origins, 25 complete blocks. Exploratory block band [-0.004, +0.011].
- **7-day K-ratio:** old/new rank IC 0.118/0.125; delta +0.007; modelling-scale MAE change -0.1%; 45 origins, 25 complete blocks. Exploratory block band [-0.015, +0.021].
- **7-day Sharpe:** old/new rank IC 0.103/0.114; delta +0.012; modelling-scale MAE change -0.1%; 45 origins, 25 complete blocks. Exploratory block band [-0.011, +0.024].
- **30-day CAGR:** old/new rank IC 0.067/0.069; delta +0.002; modelling-scale MAE change +0.3%; 24 origins, 3 complete blocks. Too few blocks for an uncertainty band.
- **30-day K-ratio:** old/new rank IC 0.159/0.166; delta +0.007; modelling-scale MAE change -0.1%; 24 origins, 3 complete blocks. Too few blocks for an uncertainty band.
- **30-day Sharpe:** old/new rank IC 0.120/0.132; delta +0.012; modelling-scale MAE change -0.0%; 24 origins, 3 complete blocks. Too few blocks for an uncertainty band.
- **90 days:** no supported paired forecast comparison. Illustrative forecasts and descriptive feature relationships do not establish model skill.

## One-year and all-history amendment

One year follows the existing 360-day convention. Max uses calendar-day NAV from the first actual positive mark through T−1, causally carrying prices over gaps, with at least three fitted marks and no backfill or gap compression. Recorded history is not certified inception. Short-window ablations keep shared missingness controls fixed, so the longer K values can be assessed separately from history availability. The previous four-window run is archived under `09-before-long-k`.

- **360d K-ratio:** 8,052/31,274 supported-anchor rows have a finite input (25.7%).
- **max K-ratio:** 30,560/31,274 supported-anchor rows have a finite input (97.7%).
- **Available max durations:** 2–1312 days, median 217 days; 18,047 of 30,560 finite max rows span at least 180 days. Max is not necessarily a longer input for young vaults. One-year availability is measured separately above; young vaults retain usable shorter inputs.
- **Existing + K, 7-day CAGR:** versus four-window K, same-row rank IC 0.0173 → 0.0214; delta +0.0040; MAE change +0.2%; 45 origins, 25 blocks.
- **Existing + short K + max, 7-day CAGR:** versus four-window K, same-row rank IC 0.0173 → 0.0221; delta +0.0047; MAE change +0.1%; 45 origins, 25 blocks.
- **Existing + short K + year, 7-day CAGR:** versus four-window K, same-row rank IC 0.0173 → 0.0164; delta -0.0009; MAE change +0.0%; 45 origins, 25 blocks.
- **Existing + K, 7-day K-ratio:** versus four-window K, same-row rank IC 0.1136 → 0.1254; delta +0.0117; MAE change -0.2%; 45 origins, 25 blocks.
- **Existing + short K + max, 7-day K-ratio:** versus four-window K, same-row rank IC 0.1136 → 0.1255; delta +0.0119; MAE change -0.2%; 45 origins, 25 blocks.
- **Existing + short K + year, 7-day K-ratio:** versus four-window K, same-row rank IC 0.1136 → 0.1133; delta -0.0003; MAE change +0.0%; 45 origins, 25 blocks.
- **Existing + K, 7-day Sharpe:** versus four-window K, same-row rank IC 0.0987 → 0.1143; delta +0.0156; MAE change -0.2%; 45 origins, 25 blocks.
- **Existing + short K + max, 7-day Sharpe:** versus four-window K, same-row rank IC 0.0987 → 0.1143; delta +0.0156; MAE change -0.2%; 45 origins, 25 blocks.
- **Existing + short K + year, 7-day Sharpe:** versus four-window K, same-row rank IC 0.0987 → 0.0986; delta -0.0001; MAE change +0.0%; 45 origins, 25 blocks.
- **Existing + K, 30-day CAGR:** versus four-window K, same-row rank IC 0.0665 → 0.0688; delta +0.0023; MAE change +0.2%; 24 origins, 3 blocks.
- **Existing + short K + max, 30-day CAGR:** versus four-window K, same-row rank IC 0.0665 → 0.0716; delta +0.0051; MAE change +0.2%; 24 origins, 3 blocks.
- **Existing + short K + year, 30-day CAGR:** versus four-window K, same-row rank IC 0.0665 → 0.0634; delta -0.0031; MAE change +0.1%; 24 origins, 3 blocks.
- **Existing + K, 30-day K-ratio:** versus four-window K, same-row rank IC 0.1592 → 0.1657; delta +0.0065; MAE change -0.1%; 24 origins, 3 blocks.
- **Existing + short K + max, 30-day K-ratio:** versus four-window K, same-row rank IC 0.1592 → 0.1661; delta +0.0069; MAE change -0.1%; 24 origins, 3 blocks.
- **Existing + short K + year, 30-day K-ratio:** versus four-window K, same-row rank IC 0.1592 → 0.1580; delta -0.0012; MAE change +0.0%; 24 origins, 3 blocks.
- **Existing + K, 30-day Sharpe:** versus four-window K, same-row rank IC 0.1192 → 0.1319; delta +0.0127; MAE change -0.0%; 24 origins, 3 blocks.
- **Existing + short K + max, 30-day Sharpe:** versus four-window K, same-row rank IC 0.1192 → 0.1327; delta +0.0136; MAE change -0.0%; 24 origins, 3 blocks.
- **Existing + short K + year, 30-day Sharpe:** versus four-window K, same-row rank IC 0.1192 → 0.1178; delta -0.0014; MAE change +0.0%; 24 origins, 3 blocks.

## Summary

The notebook covers all nine target/horizon combinations, preserves partial-history forecasts with missing=True, and retains actual observation masks alongside carried daily prices. Future labels are censored after the last actual vault observation. Horizon support is measured in the tables below. Separate linked notebooks hold the complete K scatter catalogue.

## Robustness

Independent metric and target checks, experiment-05 compatibility, paired model ablations, quarter/cohort/phase/delay sensitivities, non-overlapping origins, longer blocks and influential-vault checks are included. Current-snapshot selection and limited daily history constrain conclusions. No strategy rules are changed.

## Interpretation

The support audit controls what can be inferred. A relationship between past and future K may reflect persistent smoothness, low activity or reporting precision; it does not itself show prediction of profitable returns. The measured training and calendar-block support determines which horizons support forecast comparisons.


In [1]:
from pathlib import Path
import sys
ROOT=Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
if not (ROOT/'k_ratio_features.py').exists():
    ROOT=Path('/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd')
sys.path.insert(0,str(ROOT))
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from score_matrix_visualisation import compact_output
from k_ratio_features import *
from k_ratio_forecast_audit import audit_dataset, compatibility_ic
from k_ratio_forecast_models import run_models
from k_ratio_forecast_statistics import analyse
from k_ratio_forecast_extensions import run_extensions
from k_ratio_forecast_feedback import run_feedback
from k_ratio_forecast_report import table, show, report_overview, robustness_report, ic_map, forecast_scatter, feature_scatter, write_catalogue, write_findings, freeze_design, feedback_report


## Pinned inputs and genuine daily observations

Rebuild all metric features from actual positive NAV observations with causal forward filling on the daily grid. Cached earlier files provide vault identities and a separate source-compatibility audit, never composite-score predictors. Actual unchanged marks remain observations; synthetic carry-forward is not an actual daily mark.

In [2]:
with compact_output('Audited feature and future-label dataset — estimated 1 minute'):
    data=prepare_data()
table(pd.DataFrame([{'Vaults':data['manifest']['vaults'],'Existing primary features':len(EXISTING),'New K windows':len(K_ALL),'Forecast cadence (days)':4,'Horizons (days)':'7 / 30 / 90','Missing policy':'True (partial histories)','Publication clock':'Unknown; observation-time replay'}]),
      'Research inputs','The 45 existing features use nine lookbacks. Main K inputs are 30/60/90/180/360 days and max recorded history; the inventory includes other diagnostic windows. Five 45-day compatibility features remain audit-only. No new remote archive is fetched.')


Audited feature and future-label dataset — estimated 1 minute — complete

### Research inputs

2026-01-01 → 2026-10-05 (end exclusive, UTC). The 45 existing features use nine lookbacks. Main K inputs are 30/60/90/180/360 days and max recorded history; the inventory includes other diagnostic windows. Five 45-day compatibility features remain audit-only. No new remote archive is fetched.

Vaults,Existing primary features,New K windows,Forecast cadence (days),Horizons (days),Missing policy,Publication clock
514,45,10,4,7 / 30 / 90,True (partial histories),Unknown; observation-time replay


## Numerical and source compatibility checks

Check K against independently solved OLS, its signed-R² identity, first-return influence, scale/sign/order behaviour and missing/flat paths. Recompute forward-filled targets independently, and reproduce experiment 05's source feature values and 14/30/60-day feature IC on shared origins.

In [3]:
with compact_output('Independent metrics and experiment-05 compatibility audits — estimated 1 minute'):
    audit=audit_dataset(data)
    compatibility=compatibility_ic(data)
table(audit,'Numerical and genuine-label checks','Every boolean check must pass. Source cells are compared with the frozen 05 feature cache; sampled supported labels are recomputed through independent paths.')
table(compatibility,'Reproducing the earlier feature analysis','These old 14/30/60-day associations verify compatibility only. The experiment-09 forecast horizons remain 7/30/90 days.')


Independent metrics and experiment-05 compatibility audits — estimated 1 minute — complete

### Numerical and genuine-label checks

2026-01-01 → 2026-10-05 (end exclusive, UTC). Every boolean check must pass. Source cells are compared with the frozen 05 feature cache; sampled supported labels are recomputed through independent paths.

Check,Result
independent_OLS,True
signed_R2_identity,True
scale_invariance,True
first_return_changes_primary_K,True
ends_only_excludes_first_return,True
flat_is_undefined,True
exact_linear_log_path_is_undefined,True
negative_path_negative_K,True
general_return_permutation_changes_K,True
complete_return_reversal_preserves_anchor_K,True


### Reproducing the earlier feature analysis

2026-01-01 → 2026-10-05 (end exclusive, UTC). These old 14/30/60-day associations verify compatibility only. The experiment-09 forecast horizons remain 7/30/90 days.

Compared IC cells,Matched,Maximum absolute error,Old horizons (days)
15210,15210,0.000,14 / 30 / 60


## Chronological models and target support

Refit monthly using strictly matured labels; predict every four days. Ridge alpha is 10; raw Sharpe uses scaled pseudo-Huber delta 1.35, with one total weight per origin. Training-only robust adapters preserve NaN, bound inputs and level predictions, and abstain without usable performance inputs. Complete-case uses 30/60/90 days. Ranking companions use causal input percentiles and matured rank targets with IC only; activity and compact-control arms are fixed sensitivities.

In [4]:
with compact_output('Fixed chronological feature forecasts — estimated 3–6 minutes'):
    models=run_models(data)
with compact_output('Individual feature evidence and paired diagnostics — estimated 3–5 minutes'):
    statistics=analyse(data,models)
with compact_output('Missingness, calibration and path-quality diagnostics — estimated 1 minute'):
    extensions=run_extensions(data,models,statistics)
with compact_output('Matched cohorts, extrapolation and attrition — estimated 1 minute'):
    feedback=run_feedback(data,models,statistics)


Fixed chronological feature forecasts — estimated 3–6 minutes — complete

Individual feature evidence and paired diagnostics — estimated 3–5 minutes — complete

Missingness, calibration and path-quality diagnostics — estimated 1 minute — complete

Matched cohorts, extrapolation and attrition — estimated 1 minute — complete

## Availability and incremental prediction

The following sections separate observation coverage, operational eligibility and predictive accuracy. The principal comparison adds six K inputs (30/60/90/180/360 days and max recorded history). Separate same-row ablations compare year/max additions with the original four-window K inventory, keeping model settings and shared availability controls fixed.

In [5]:
report_overview(data,models,statistics)


### Forecast population versus funding feasibility

2026-01-01 → 2026-10-05 (end exclusive, UTC). Primary predictors use observed NAV and age/quality only. Current guard/permission checks are separate diagnostic assumptions.

Population,Vault/date rows,Distinct vaults
Scheduled current-snapshot identities,35980,514
Supported daily NAV anchor (may be carried),31274,514
05 guard/gate eligible (current policy),14850,444
Permission observation clock unknown,14978,514
Eligibility + deposit + 90d sizing history (diagnostic),10526,376
05 source-equivalent baseline requested positive targets,420,35


### Future label support by target and horizon

2026-01-01 → 2026-10-05 (end exclusive, UTC). All targets use causal forward-filled daily NAV; no backfill before first mark or target after last actual mark. Administrative censoring and undefined ratios are not zero outcomes.

horizon,target,scheduled_origins,valid_origins,observed_anchor_rows,valid_labels,label_coverage,right_censored_rows,complete_daily_paths
7,CAGR,70,68,31274,26543,84.9%,1028,26543
7,Sharpe,70,68,31274,19965,63.8%,1028,26543
7,K-ratio,70,68,31274,19965,63.8%,1028,26543
30,CAGR,70,62,31274,23514,75.2%,4103,23514
30,Sharpe,70,62,31274,19639,62.8%,4103,23514
30,K-ratio,70,62,31274,19938,63.8%,4103,23514
90,CAGR,70,47,31274,16416,52.5%,11653,16416
90,Sharpe,70,47,31274,14947,47.8%,11653,16416
90,K-ratio,70,47,31274,15075,48.2%,11653,16416


### Sparse NAV support and observation quality

2026-01-01 → 2026-10-05 (end exclusive, UTC). Forward filling carries the latest known price; it does not add observations. Actual mark counts and ages remain quality controls. No leading backfill or future labels beyond the last actual vault mark are permitted.

...

### K incremental rank IC

2026-01-01 → 2026-10-05 (end exclusive, UTC). Columns are the requested horizons. These are paired changes, not independently selected winners; undefined cells reflect lack of model support.

### K proportional change in forecast MAE

2026-01-01 → 2026-10-05 (end exclusive, UTC). Columns are the requested horizons. These are paired changes, not independently selected winners; undefined cells reflect lack of model support.

### 30-day feature redundancy

2026-01-01 → 2026-10-05 (end exclusive, UTC). Correlation between historical inputs is not prediction of future performance. K and signed R² have an exact fixed-sample ordering identity; slope and residual scale help interpret it.

### 90-day feature redundancy

2026-01-01 → 2026-10-05 (end exclusive, UTC). Correlation between historical inputs is not prediction of future performance. K and signed R² have an exact fixed-sample ordering identity; slope and residual scale help interpret it.

### 180-day feature redundancy

2026-01-01 → 2026-10-05 (end exclusive, UTC). Correlation between historical inputs is not prediction of future performance. K and signed R² have an exact fixed-sample ordering identity; slope and residual scale help interpret it.

## Forecasting future CAGR

Heatmaps show individual feature associations. Scatter plots show chronological Existing + K predictions in target units where supported, coloured by origin quarter. CAGR is fitted and shown as h-day log return so short-horizon annualisation does not overwhelm the chart; it has the same ordering as CAGR. Future labels and errors retain all tails.

In [6]:
show(ic_map(statistics,7,'CAGR'),
    'Individual features versus 7-day future CAGR',
    'Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.')
p=models['predictions']
supported=p[p.horizon.eq(7)&p.target.eq('CAGR')&p.arm.eq('Existing + K')&p.supported&p.phase.eq(0)&p.delay.eq(0)&~p.source&np.isfinite(p.actual)]
if len(supported):
    show(forecast_scatter(models,7,'CAGR'),
        '7-day CAGR: chronological forecast versus outcome',
        'One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.')
else:
    display(Markdown('### 7-day CAGR: forecast support\n\n'+PERIOD+'. No supported target-valid model predictions. Required label maturation and training support are insufficient; this is not evidence that the feature relationship is zero.'))
    show(forecast_scatter(models,7,'CAGR',supported_only=False),
        '7-day CAGR: illustrative thin-support forecasts',
        'These forecasts fail the predeclared training-support gate. They are shown to inspect the requested horizon, excluded from headline errors/IC and cannot establish forecasting skill. Colours are forecast-origin quarters.')


### Individual features versus 7-day future CAGR

2026-01-01 → 2026-10-05 (end exclusive, UTC). Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.

### 7-day CAGR: chronological forecast versus outcome

2026-01-01 → 2026-10-05 (end exclusive, UTC). One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.

In [7]:
show(ic_map(statistics,30,'CAGR'),
    'Individual features versus 30-day future CAGR',
    'Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.')
p=models['predictions']
supported=p[p.horizon.eq(30)&p.target.eq('CAGR')&p.arm.eq('Existing + K')&p.supported&p.phase.eq(0)&p.delay.eq(0)&~p.source&np.isfinite(p.actual)]
if len(supported):
    show(forecast_scatter(models,30,'CAGR'),
        '30-day CAGR: chronological forecast versus outcome',
        'One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.')
else:
    display(Markdown('### 30-day CAGR: forecast support\n\n'+PERIOD+'. No supported target-valid model predictions. Required label maturation and training support are insufficient; this is not evidence that the feature relationship is zero.'))
    show(forecast_scatter(models,30,'CAGR',supported_only=False),
        '30-day CAGR: illustrative thin-support forecasts',
        'These forecasts fail the predeclared training-support gate. They are shown to inspect the requested horizon, excluded from headline errors/IC and cannot establish forecasting skill. Colours are forecast-origin quarters.')


### Individual features versus 30-day future CAGR

2026-01-01 → 2026-10-05 (end exclusive, UTC). Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.

### 30-day CAGR: chronological forecast versus outcome

2026-01-01 → 2026-10-05 (end exclusive, UTC). One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.

In [8]:
show(ic_map(statistics,90,'CAGR'),
    'Individual features versus 90-day future CAGR',
    'Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.')
p=models['predictions']
supported=p[p.horizon.eq(90)&p.target.eq('CAGR')&p.arm.eq('Existing + K')&p.supported&p.phase.eq(0)&p.delay.eq(0)&~p.source&np.isfinite(p.actual)]
if len(supported):
    show(forecast_scatter(models,90,'CAGR'),
        '90-day CAGR: chronological forecast versus outcome',
        'One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.')
else:
    display(Markdown('### 90-day CAGR: forecast support\n\n'+PERIOD+'. No supported target-valid model predictions. Required label maturation and training support are insufficient; this is not evidence that the feature relationship is zero.'))
    show(forecast_scatter(models,90,'CAGR',supported_only=False),
        '90-day CAGR: illustrative thin-support forecasts',
        'These forecasts fail the predeclared training-support gate. They are shown to inspect the requested horizon, excluded from headline errors/IC and cannot establish forecasting skill. Colours are forecast-origin quarters.')


### Individual features versus 90-day future CAGR

2026-01-01 → 2026-10-05 (end exclusive, UTC). Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.

### 90-day CAGR: forecast support

2026-01-01 → 2026-10-05 (end exclusive, UTC). No supported target-valid model predictions. Required label maturation and training support are insufficient; this is not evidence that the feature relationship is zero.

### 90-day CAGR: illustrative thin-support forecasts

2026-01-01 → 2026-10-05 (end exclusive, UTC). These forecasts fail the predeclared training-support gate. They are shown to inspect the requested horizon, excluded from headline errors/IC and cannot establish forecasting skill. Colours are forecast-origin quarters.

## Forecasting future Sharpe

Heatmaps show individual feature associations. Scatter plots show chronological Existing + K predictions in target units where supported, coloured by origin quarter. CAGR is fitted and shown as h-day log return so short-horizon annualisation does not overwhelm the chart; it has the same ordering as CAGR. Future labels and errors retain all tails.

In [9]:
show(ic_map(statistics,7,'Sharpe'),
    'Individual features versus 7-day future Sharpe',
    'Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.')
p=models['predictions']
supported=p[p.horizon.eq(7)&p.target.eq('Sharpe')&p.arm.eq('Existing + K')&p.supported&p.phase.eq(0)&p.delay.eq(0)&~p.source&np.isfinite(p.actual)]
if len(supported):
    show(forecast_scatter(models,7,'Sharpe'),
        '7-day Sharpe: chronological forecast versus outcome',
        'One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.')
else:
    display(Markdown('### 7-day Sharpe: forecast support\n\n'+PERIOD+'. No supported target-valid model predictions. Required label maturation and training support are insufficient; this is not evidence that the feature relationship is zero.'))
    show(forecast_scatter(models,7,'Sharpe',supported_only=False),
        '7-day Sharpe: illustrative thin-support forecasts',
        'These forecasts fail the predeclared training-support gate. They are shown to inspect the requested horizon, excluded from headline errors/IC and cannot establish forecasting skill. Colours are forecast-origin quarters.')


### Individual features versus 7-day future Sharpe

2026-01-01 → 2026-10-05 (end exclusive, UTC). Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.

### 7-day Sharpe: chronological forecast versus outcome

2026-01-01 → 2026-10-05 (end exclusive, UTC). One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.

In [10]:
show(ic_map(statistics,30,'Sharpe'),
    'Individual features versus 30-day future Sharpe',
    'Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.')
p=models['predictions']
supported=p[p.horizon.eq(30)&p.target.eq('Sharpe')&p.arm.eq('Existing + K')&p.supported&p.phase.eq(0)&p.delay.eq(0)&~p.source&np.isfinite(p.actual)]
if len(supported):
    show(forecast_scatter(models,30,'Sharpe'),
        '30-day Sharpe: chronological forecast versus outcome',
        'One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.')
else:
    display(Markdown('### 30-day Sharpe: forecast support\n\n'+PERIOD+'. No supported target-valid model predictions. Required label maturation and training support are insufficient; this is not evidence that the feature relationship is zero.'))
    show(forecast_scatter(models,30,'Sharpe',supported_only=False),
        '30-day Sharpe: illustrative thin-support forecasts',
        'These forecasts fail the predeclared training-support gate. They are shown to inspect the requested horizon, excluded from headline errors/IC and cannot establish forecasting skill. Colours are forecast-origin quarters.')


### Individual features versus 30-day future Sharpe

2026-01-01 → 2026-10-05 (end exclusive, UTC). Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.

### 30-day Sharpe: chronological forecast versus outcome

2026-01-01 → 2026-10-05 (end exclusive, UTC). One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.

In [11]:
show(ic_map(statistics,90,'Sharpe'),
    'Individual features versus 90-day future Sharpe',
    'Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.')
p=models['predictions']
supported=p[p.horizon.eq(90)&p.target.eq('Sharpe')&p.arm.eq('Existing + K')&p.supported&p.phase.eq(0)&p.delay.eq(0)&~p.source&np.isfinite(p.actual)]
if len(supported):
    show(forecast_scatter(models,90,'Sharpe'),
        '90-day Sharpe: chronological forecast versus outcome',
        'One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.')
else:
    display(Markdown('### 90-day Sharpe: forecast support\n\n'+PERIOD+'. No supported target-valid model predictions. Required label maturation and training support are insufficient; this is not evidence that the feature relationship is zero.'))
    show(forecast_scatter(models,90,'Sharpe',supported_only=False),
        '90-day Sharpe: illustrative thin-support forecasts',
        'These forecasts fail the predeclared training-support gate. They are shown to inspect the requested horizon, excluded from headline errors/IC and cannot establish forecasting skill. Colours are forecast-origin quarters.')


### Individual features versus 90-day future Sharpe

2026-01-01 → 2026-10-05 (end exclusive, UTC). Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.

### 90-day Sharpe: forecast support

2026-01-01 → 2026-10-05 (end exclusive, UTC). No supported target-valid model predictions. Required label maturation and training support are insufficient; this is not evidence that the feature relationship is zero.

### 90-day Sharpe: illustrative thin-support forecasts

2026-01-01 → 2026-10-05 (end exclusive, UTC). These forecasts fail the predeclared training-support gate. They are shown to inspect the requested horizon, excluded from headline errors/IC and cannot establish forecasting skill. Colours are forecast-origin quarters.

## Forecasting future K-ratio

Heatmaps show individual feature associations. Scatter plots show chronological Existing + K predictions in target units where supported, coloured by origin quarter. CAGR is fitted and shown as h-day log return so short-horizon annualisation does not overwhelm the chart; it has the same ordering as CAGR. Future labels and errors retain all tails.

In [12]:
show(ic_map(statistics,7,'K-ratio'),
    'Individual features versus 7-day future K-ratio',
    'Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.')
p=models['predictions']
supported=p[p.horizon.eq(7)&p.target.eq('K-ratio')&p.arm.eq('Existing + K')&p.supported&p.phase.eq(0)&p.delay.eq(0)&~p.source&np.isfinite(p.actual)]
if len(supported):
    show(forecast_scatter(models,7,'K-ratio'),
        '7-day K-ratio: chronological forecast versus outcome',
        'One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.')
else:
    display(Markdown('### 7-day K-ratio: forecast support\n\n'+PERIOD+'. No supported target-valid model predictions. Required label maturation and training support are insufficient; this is not evidence that the feature relationship is zero.'))
    show(forecast_scatter(models,7,'K-ratio',supported_only=False),
        '7-day K-ratio: illustrative thin-support forecasts',
        'These forecasts fail the predeclared training-support gate. They are shown to inspect the requested horizon, excluded from headline errors/IC and cannot establish forecasting skill. Colours are forecast-origin quarters.')


### Individual features versus 7-day future K-ratio

2026-01-01 → 2026-10-05 (end exclusive, UTC). Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.

### 7-day K-ratio: chronological forecast versus outcome

2026-01-01 → 2026-10-05 (end exclusive, UTC). One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.

In [13]:
show(ic_map(statistics,30,'K-ratio'),
    'Individual features versus 30-day future K-ratio',
    'Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.')
p=models['predictions']
supported=p[p.horizon.eq(30)&p.target.eq('K-ratio')&p.arm.eq('Existing + K')&p.supported&p.phase.eq(0)&p.delay.eq(0)&~p.source&np.isfinite(p.actual)]
if len(supported):
    show(forecast_scatter(models,30,'K-ratio'),
        '30-day K-ratio: chronological forecast versus outcome',
        'One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.')
else:
    display(Markdown('### 30-day K-ratio: forecast support\n\n'+PERIOD+'. No supported target-valid model predictions. Required label maturation and training support are insufficient; this is not evidence that the feature relationship is zero.'))
    show(forecast_scatter(models,30,'K-ratio',supported_only=False),
        '30-day K-ratio: illustrative thin-support forecasts',
        'These forecasts fail the predeclared training-support gate. They are shown to inspect the requested horizon, excluded from headline errors/IC and cannot establish forecasting skill. Colours are forecast-origin quarters.')


### Individual features versus 30-day future K-ratio

2026-01-01 → 2026-10-05 (end exclusive, UTC). Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.

### 30-day K-ratio: chronological forecast versus outcome

2026-01-01 → 2026-10-05 (end exclusive, UTC). One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.

In [14]:
show(ic_map(statistics,90,'K-ratio'),
    'Individual features versus 90-day future K-ratio',
    'Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.')
p=models['predictions']
supported=p[p.horizon.eq(90)&p.target.eq('K-ratio')&p.arm.eq('Existing + K')&p.supported&p.phase.eq(0)&p.delay.eq(0)&~p.source&np.isfinite(p.actual)]
if len(supported):
    show(forecast_scatter(models,90,'K-ratio'),
        '90-day K-ratio: chronological forecast versus outcome',
        'One dot is a supported vault/origin pair. The dashed identity is a perfect forecast; the decile-median line is descriptive. Display tails are disclosed, with all data retained for calculation. Axis scales are stated on the chart; CAGR uses linear horizon log-return units.')
else:
    display(Markdown('### 90-day K-ratio: forecast support\n\n'+PERIOD+'. No supported target-valid model predictions. Required label maturation and training support are insufficient; this is not evidence that the feature relationship is zero.'))
    show(forecast_scatter(models,90,'K-ratio',supported_only=False),
        '90-day K-ratio: illustrative thin-support forecasts',
        'These forecasts fail the predeclared training-support gate. They are shown to inspect the requested horizon, excluded from headline errors/IC and cannot establish forecasting skill. Colours are forecast-origin quarters.')


### Individual features versus 90-day future K-ratio

2026-01-01 → 2026-10-05 (end exclusive, UTC). Tiles are equal-origin rank IC, with defined-origin counts. Higher historical signed drawdown means a smaller loss. Correlation is descriptive and blank cells do not mean zero skill.

### 90-day K-ratio: forecast support

2026-01-01 → 2026-10-05 (end exclusive, UTC). No supported target-valid model predictions. Required label maturation and training support are insufficient; this is not evidence that the feature relationship is zero.

### 90-day K-ratio: illustrative thin-support forecasts

2026-01-01 → 2026-10-05 (end exclusive, UTC). These forecasts fail the predeclared training-support gate. They are shown to inspect the requested horizon, excluded from headline errors/IC and cannot establish forecasting skill. Colours are forecast-origin quarters.

## Reading individual K inputs

These representative 30-day K plots show raw feature values against each future outcome at 30 days, using log-return units for CAGR. The linked catalogue contains every inventoried K window and all nine target/horizon combinations, each as a separate image.

In [15]:
for target in TARGETS:
    show(feature_scatter(data,'30d K-ratio',30,target),
         '30-day historical K versus 30-day future '+target,
         'X is historical K, not a composite score. Colours are origin quarters. Axis scales are stated on the chart; CAGR uses horizon log return; decile medians describe the relationship. Counts exclude only undefined feature/target pairs.')


### 30-day historical K versus 30-day future CAGR

2026-01-01 → 2026-10-05 (end exclusive, UTC). X is historical K, not a composite score. Colours are origin quarters. Axis scales are stated on the chart; CAGR uses horizon log return; decile medians describe the relationship. Counts exclude only undefined feature/target pairs.

### 30-day historical K versus 30-day future Sharpe

2026-01-01 → 2026-10-05 (end exclusive, UTC). X is historical K, not a composite score. Colours are origin quarters. Axis scales are stated on the chart; CAGR uses horizon log return; decile medians describe the relationship. Counts exclude only undefined feature/target pairs.

### 30-day historical K versus 30-day future K-ratio

2026-01-01 → 2026-10-05 (end exclusive, UTC). X is historical K, not a composite score. Colours are origin quarters. Axis scales are stated on the chart; CAGR uses horizon log return; decile medians describe the relationship. Counts exclude only undefined feature/target pairs.

## Baselines, missing policies and robustness

Compare forecast errors with historical constants and own-target persistence; then examine calendar phases, delayed entry, source-price filling, age/quality cohorts and concentration. All these are fixed diagnostics, not another parameter search.

In [16]:
primary=statistics['model_summary']
primary=primary[primary.phase.eq(0)&primary.delay.eq(0)&~primary.source]
for target in TARGETS:
    table(primary[primary.target.eq(target)][['horizon','arm','mean_IC','MAE','MSE','raw_MAE','demeaned_MSE','evaluation_origins','defined_IC_origins']],
          target+' forecast methods side by side',
          'Errors average within dates and then across dates. CAGR error uses h-day log return; K error uses fold-scaled asinh, with raw-unit MAE alongside. Different model coverage is disclosed; paired K effects are evaluated separately.',limit=60)
table(statistics['model_coverage'].query('supported == True'),'Forecast availability and partial histories',
      'Missing=True uses available shorter inputs. Complete-case uses the supported fixed 30/60/90-day inventory. Partial history is measured across ≤180-day existing inputs; 270/360/730-day columns are excluded from this partial-history definition. Coverage is separate from accuracy.',limit=60)
table(statistics['cohorts'],'Young and high-quality cohort comparisons',
      'Paired changes use identical vault/date observations within each cohort; small or constant cross-sections have undefined IC.')
paired=statistics['paired_summary']
table(paired[paired.old.eq('Existing')][['horizon','target','phase','delay','source','mean_delta_IC','MAE_change_fraction','calendar_blocks']],
      'Cadence, delayed-entry and source-price sensitivities',
      'phase 0 is the Jan-1 four-day schedule; phases 1–3 shift origins. delay=1 shifts both target endpoints one day. source=True changes only existing feature prices, keeping actual future labels.',limit=60)
table(statistics['blocks'].query('phase == 0 and delay == 0 and source == False and old == "Existing"'),
      'Independent-calendar-block paired changes','Each row is one complete horizon-length calendar block. Below eight usable blocks, show signs and estimates without an uncertainty interval.',limit=60)
robustness_report(data,models,statistics,extensions)
feedback_report(data,models,statistics,extensions,feedback)


### CAGR forecast methods side by side

2026-01-01 → 2026-10-05 (end exclusive, UTC). Errors average within dates and then across dates. CAGR error uses h-day log return; K error uses fold-scaled asinh, with raw-unit MAE alongside. Different model coverage is disclosed; paired K effects are evaluated separately.

horizon,arm,mean_IC,MAE,MSE,raw_MAE,demeaned_MSE,evaluation_origins,defined_IC_origins
7,Activity existing,0.0223,0.099019,0.196434,0.099019,0.195091,45,45
7,Activity existing + K,0.0314,0.100641,0.196342,0.100641,0.194987,45,45
7,Compact,0.0049,0.098230,0.200541,0.098230,0.199069,45,45
7,Compact + K,0.0292,0.099639,0.198065,0.099639,0.196663,45,45
7,Compact controls existing,0.0206,0.098926,0.196186,0.098926,0.194780,45,45
7,Compact controls existing + K,0.0329,0.100469,0.195212,0.100469,0.193797,45,45
7,Constant,—,0.101241,0.198537,0.101241,0.197342,45,0
7,Decomposition,0.0284,0.098869,0.195267,0.098869,0.193855,45,45
7,Existing,0.0185,0.099141,0.196141,0.099141,0.194751,45,45
7,Existing + K,0.0333,0.100983,0.195147,0.100983,0.193757,45,45


### Sharpe forecast methods side by side

2026-01-01 → 2026-10-05 (end exclusive, UTC). Errors average within dates and then across dates. CAGR error uses h-day log return; K error uses fold-scaled asinh, with raw-unit MAE alongside. Different model coverage is disclosed; paired K effects are evaluated separately.

horizon,arm,mean_IC,MAE,MSE,raw_MAE,demeaned_MSE,evaluation_origins,defined_IC_origins
7,Activity existing,0.0668,6.989734,79.009271,6.989734,74.523433,38,38
7,Activity existing + K,0.0728,6.995233,79.182160,6.995233,74.736849,38,38
7,Compact,0.1141,6.884403,73.598554,6.884403,70.280469,45,45
7,Compact + K,0.1253,6.877889,73.472286,6.877889,70.247876,45,45
7,Compact controls existing,0.1036,6.889025,73.559491,6.889025,70.298876,45,45
7,Compact controls existing + K,0.1190,6.885511,73.513652,6.885511,70.313287,45,45
7,Constant,—,6.930462,74.623236,6.930462,71.495512,45,0
7,Decomposition,0.1089,6.883807,73.501788,6.883807,70.199891,45,45
7,Existing,0.1026,6.888747,73.600863,6.888747,70.331389,45,45
7,Existing + K,0.1172,6.884847,73.537835,6.884847,70.326904,45,45


### K-ratio forecast methods side by side

2026-01-01 → 2026-10-05 (end exclusive, UTC). Errors average within dates and then across dates. CAGR error uses h-day log return; K error uses fold-scaled asinh, with raw-unit MAE alongside. Different model coverage is disclosed; paired K effects are evaluated separately.

...

feature,horizon,target,quintile,median_outcome,p10_outcome,p90_outcome,origins,paired_rows
30d K-ratio,30,Sharpe,1,-0.274,-4.830,4.016,62,3485
30d K-ratio,30,Sharpe,2,0.251,-4.485,4.332,62,3505
30d K-ratio,30,Sharpe,3,0.505,-4.344,4.618,62,3501
30d K-ratio,30,Sharpe,4,0.418,-4.380,5.027,62,3506
30d K-ratio,30,Sharpe,5,0.596,-4.165,5.820,62,3537
90d K-ratio,30,Sharpe,1,-0.269,-4.797,3.746,62,2957
90d K-ratio,30,Sharpe,2,0.382,-3.983,4.159,62,2982
90d K-ratio,30,Sharpe,3,0.051,-4.600,4.459,62,2975
90d K-ratio,30,Sharpe,4,0.264,-4.699,5.264,62,2982
90d K-ratio,30,Sharpe,5,0.666,-4.251,5.669,62,3003


### 30-day K-ratio: outcome distributions by historical input quintile

2026-01-01 → 2026-10-05 (end exclusive, UTC). Within-origin tie-aware quintiles; average date medians and 10th/90th percentiles. CAGR uses horizon log return. Different features may have different available cohorts.

feature,horizon,target,quintile,median_outcome,p10_outcome,p90_outcome,origins,paired_rows
30d K-ratio,30,K-ratio,1,-0.719,-5.963,4.460,62,3544
30d K-ratio,30,K-ratio,2,-0.107,-5.345,4.534,62,3569
30d K-ratio,30,K-ratio,3,0.046,-5.214,5.229,62,3559
30d K-ratio,30,K-ratio,4,-0.118,-5.454,5.885,62,3566
30d K-ratio,30,K-ratio,5,0.487,-4.948,7.718,62,3595
90d K-ratio,30,K-ratio,1,-0.705,-5.936,3.859,62,3006
90d K-ratio,30,K-ratio,2,-0.093,-4.765,4.655,62,3038
90d K-ratio,30,K-ratio,3,-0.310,-5.265,4.933,62,3033
90d K-ratio,30,K-ratio,4,0.011,-5.619,5.925,62,3037
90d K-ratio,30,K-ratio,5,0.369,-5.012,6.999,62,3056


### Unfloored Calmar tail and availability

2026-01-01 → 2026-10-05 (end exclusive, UTC). The existing Calmar floor is retained for compatibility. This diagnostic exposes the numerical tails; robust transforms and bounds act on all model inputs.

days,finite_floored,finite_unfloored,unfloored_median,unfloored_p99,unfloored_max
30,29454,21380,-1.040,66471.407,1417628170136.823
90,25347,20970,-1.011,319.006,15187570.014
180,18129,16986,-0.922,60.414,652324.727


## Secondary earlier-history evidence

This section is explicitly outside the default 2026 report. Earlier labels must meet the same actual-observation rules; no interpolation creates supervised Sharpe/K history.

In [17]:
prior=statistics['feature_summary'].query('period == "Long prior"')
table(prior.groupby(['horizon','target']).agg(defined_feature_origin_cells=('defined_origins','sum'),max_defined_origins=('defined_origins','max')).reset_index(),
      'Earlier actual-label feature support','This is a coverage audit, not extra independent model-validation evidence.',
      period='2023-03-01 → 2026-01-01 (end exclusive, UTC)')


### Earlier actual-label feature support

2023-03-01 → 2026-01-01 (end exclusive, UTC). This is a coverage audit, not extra independent model-validation evidence.

horizon,target,defined_feature_origin_cells,max_defined_origins
7,CAGR,7002,162
7,K-ratio,6177,149
7,Sharpe,6177,149
30,CAGR,7002,162
30,K-ratio,6810,155
30,Sharpe,6810,155
90,CAGR,7002,162
90,K-ratio,6825,159
90,Sharpe,6825,159


## Complete individual K scatter catalogue

Each linked notebook has nine target/horizon sections, each with its own heading and description. Available combinations have separate images; unsupported long lookbacks have explanations. The 45-day notebook is an endpoint/legacy compatibility sensitivity. Splitting the catalogue keeps the main notebook renderable in VS Code.

In [18]:
with compact_output('Separate K scatter catalogue — estimated 1 minute'):
    catalogue=write_catalogue(data)
display(Markdown(catalogue))


Separate K scatter catalogue — estimated 1 minute — complete

- [30d K-ratio: nine target/horizon sections](09-feature-scatter-catalogue/09-features-30d.ipynb)
- [46d K-ratio: nine target/horizon sections](09-feature-scatter-catalogue/09-features-46d.ipynb)
- [60d K-ratio: nine target/horizon sections](09-feature-scatter-catalogue/09-features-60d.ipynb)
- [90d K-ratio: nine target/horizon sections](09-feature-scatter-catalogue/09-features-90d.ipynb)
- [120d K-ratio: nine target/horizon sections](09-feature-scatter-catalogue/09-features-120d.ipynb)
- [180d K-ratio: nine target/horizon sections](09-feature-scatter-catalogue/09-features-180d.ipynb)
- [270d K-ratio: nine target/horizon sections](09-feature-scatter-catalogue/09-features-270d.ipynb)
- [360d K-ratio: nine target/horizon sections](09-feature-scatter-catalogue/09-features-360d.ipynb)
- [730d K-ratio: nine target/horizon sections](09-feature-scatter-catalogue/09-features-730d.ipynb)
- [max K-ratio: nine target/horizon sections](09-feature-scatter-catalogue/09-features-max.ipynb)
- [45d K-ratio: nine target/horizon sections](09-feature-scatter-catalogue/09-features-45d.ipynb)

## Reproducibility and research conclusions

Complete datasets, folds, support tables, numerical audits and individual PNG exports live under the ignored experiment-09 artefact namespace. The written findings distinguish incremental information, persistence and lack of evidence.

In [19]:
table(freeze_design(data),'Frozen prospective confirmation windows','The next four-day phase begins after the design date. These are earliest dates for eight complete horizon-length blocks and matured labels, conditional on observation and model support; eight blocks do not guarantee reliable inference.')
findings=write_findings(statistics,models,data,extensions,feedback)
display(Markdown(findings.replace('# Experiment 09 — K-ratio feature forecast findings','### Measured findings',1)))
table(pd.DataFrame([{'File':path.name,'SHA-256':sha256(path)} for path in sorted(ROOT.glob('k_ratio_*.py'))]),
      'Helper source fingerprints','Together with the pinned archive hash and model/data manifests, these identify the exact experiment implementation.')


### Frozen prospective confirmation windows

2026-01-01 → 2026-10-05 (end exclusive, UTC). The next four-day phase begins after the design date. These are earliest dates for eight complete horizon-length blocks and matured labels, conditional on observation and model support; eight blocks do not guarantee reliable inference.

horizon,first_origin,origin_window_end_exclusive,last_scheduled_origin,earliest_all_labels_mature,required_complete_blocks,block_days
7,2026-10-12 00:00:00,2026-12-07 00:00:00,2026-12-03 00:00:00,2026-12-10 00:00:00,8,7
30,2026-10-12 00:00:00,2027-06-09 00:00:00,2027-06-05 00:00:00,2027-07-05 00:00:00,8,30
90,2026-10-12 00:00:00,2028-10-01 00:00:00,2028-09-27 00:00:00,2028-12-26 00:00:00,8,90


### Measured findings

Default period: **2026-01-01 → 2026-10-05 (end exclusive, UTC)**. In-place post-review amendment; original results archived separately. No data refresh or portfolio-rule change.

## Key new insights

All historical metrics and realised targets now use causal daily forward fill; the unchanged archive and pre-fill results are preserved in `09-before-sparse-fill`. Adding six K inputs (30/60/90/180/360 days and max recorded history) improves primary mean rank IC in **6 of 6** supported target/horizon comparisons. This is a measured result of fixed amended models on previously researched history, not a general claim about K or an untouched holdout.

- **7-day CAGR:** Existing/Existing + K rank IC 0.018/0.021; delta +0.003; modelling-scale MAE change +0.5%; 45 origins, 25 complete blocks. Exploratory block band [-0.004, +0.011].
- **7-day K-ratio:** Existing/Existing + K rank IC 0.118/0.125; delta +0.007; modelling-scale MAE change -0.1%; 45 origins, 25 complete blocks. Exploratory block band [-0.015, +0.021].
- **7-day Sharpe:** Existing/Existing + K rank IC 0.103/0.114; delta +0.012; modelling-scale MAE change -0.1%; 45 origins, 25 complete blocks. Exploratory block band [-0.011, +0.024].
- **30-day CAGR:** Existing/Existing + K rank IC 0.067/0.069; delta +0.002; modelling-scale MAE change +0.3%; 24 origins, 3 complete blocks. Too few blocks for an uncertainty band.
- **30-day K-ratio:** Existing/Existing + K rank IC 0.159/0.166; delta +0.007; modelling-scale MAE change -0.1%; 24 origins, 3 complete blocks. Too few blocks for an uncertainty band.
- **30-day Sharpe:** Existing/Existing + K rank IC 0.120/0.132; delta +0.012; modelling-scale MAE change -0.0%; 24 origins, 3 complete blocks. Too few blocks for an uncertainty band.
- **90 days:** no supported paired forecast comparison. Illustrative forecasts and descriptive feature relationships do not establish model skill.

## One-year and all-history amendment

One year follows the existing 360-day convention. Max uses calendar-day NAV from the first actual positive mark through T−1, causally carrying prices over gaps, with at least three fitted marks and no backfill or gap compression. Recorded history is not certified inception. Short-window ablations keep shared missingness controls fixed, so the longer K values can be assessed separately from history availability. The previous four-window run is archived under `09-before-long-k`.

- **360d K-ratio:** 8,052/31,274 supported-anchor rows have a finite input (25.7%).
- **max K-ratio:** 30,560/31,274 supported-anchor rows have a finite input (97.7%).
- **Available max durations:** 2–1312 days, median 217 days; 18,047 of 30,560 finite max rows span at least 180 days. Max is not necessarily a longer input for young vaults. One-year availability is measured separately above; young vaults retain usable shorter inputs.
- **Existing + K, 7-day CAGR:** versus four-window K, same-row rank IC 0.0173 → 0.0214; delta +0.0040; MAE change +0.2%; 45 origins, 25 blocks.
- **Existing + short K + max, 7-day CAGR:** versus four-window K, same-row rank IC 0.0173 → 0.0221; delta +0.0047; MAE change +0.1%; 45 origins, 25 blocks.
- **Existing + short K + year, 7-day CAGR:** versus four-window K, same-row rank IC 0.0173 → 0.0164; delta -0.0009; MAE change +0.0%; 45 origins, 25 blocks.
- **Existing + K, 7-day K-ratio:** versus four-window K, same-row rank IC 0.1136 → 0.1254; delta +0.0117; MAE change -0.2%; 45 origins, 25 blocks.
- **Existing + short K + max, 7-day K-ratio:** versus four-window K, same-row rank IC 0.1136 → 0.1255; delta +0.0119; MAE change -0.2%; 45 origins, 25 blocks.
- **Existing + short K + year, 7-day K-ratio:** versus four-window K, same-row rank IC 0.1136 → 0.1133; delta -0.0003; MAE change +0.0%; 45 origins, 25 blocks.
- **Existing + K, 7-day Sharpe:** versus four-window K, same-row rank IC 0.0987 → 0.1143; delta +0.0156; MAE change -0.2%; 45 origins, 25 blocks.
- **Existing + short K + max, 7-day Sharpe:** versus four-window K, same-row rank IC 0.0987 → 0.1143; delta +0.0156; MAE change -0.2%; 45 origins, 25 blocks.
- **Existing + short K + year, 7-day Sharpe:** versus four-window K, same-row rank IC 0.0987 → 0.0986; delta -0.0001; MAE change +0.0%; 45 origins, 25 blocks.
- **Existing + K, 30-day CAGR:** versus four-window K, same-row rank IC 0.0665 → 0.0688; delta +0.0023; MAE change +0.2%; 24 origins, 3 blocks.
- **Existing + short K + max, 30-day CAGR:** versus four-window K, same-row rank IC 0.0665 → 0.0716; delta +0.0051; MAE change +0.2%; 24 origins, 3 blocks.
- **Existing + short K + year, 30-day CAGR:** versus four-window K, same-row rank IC 0.0665 → 0.0634; delta -0.0031; MAE change +0.1%; 24 origins, 3 blocks.
- **Existing + K, 30-day K-ratio:** versus four-window K, same-row rank IC 0.1592 → 0.1657; delta +0.0065; MAE change -0.1%; 24 origins, 3 blocks.
- **Existing + short K + max, 30-day K-ratio:** versus four-window K, same-row rank IC 0.1592 → 0.1661; delta +0.0069; MAE change -0.1%; 24 origins, 3 blocks.
- **Existing + short K + year, 30-day K-ratio:** versus four-window K, same-row rank IC 0.1592 → 0.1580; delta -0.0012; MAE change +0.0%; 24 origins, 3 blocks.
- **Existing + K, 30-day Sharpe:** versus four-window K, same-row rank IC 0.1192 → 0.1319; delta +0.0127; MAE change -0.0%; 24 origins, 3 blocks.
- **Existing + short K + max, 30-day Sharpe:** versus four-window K, same-row rank IC 0.1192 → 0.1327; delta +0.0136; MAE change -0.0%; 24 origins, 3 blocks.
- **Existing + short K + year, 30-day Sharpe:** versus four-window K, same-row rank IC 0.1192 → 0.1178; delta -0.0014; MAE change +0.0%; 24 origins, 3 blocks.

## Sensitivities

- **Compact:** positive K delta in 6/6 supported cells; observed delta range +0.006 to +0.011. This range is not a confidence bound.
- **Rank existing:** positive K delta in 6/6 supported cells; observed delta range +0.002 to +0.011. This range is not a confidence bound.
- **Activity existing:** positive K delta in 3/6 supported cells; observed delta range -0.002 to +0.010. This range is not a confidence bound.
- **Compact controls existing:** positive K delta in 6/6 supported cells; observed delta range +0.002 to +0.013. This range is not a confidence bound.
- **Existing complete-case:** positive K delta in 0/6 supported cells; observed delta range -0.010 to -0.002. This range is not a confidence bound.
  - 7-day CAGR: 45 target-valid paired origins, 45 defined IC origins. Sparse later-calendar subsets cannot confirm incremental skill.
  - 7-day K-ratio: 45 target-valid paired origins, 45 defined IC origins. Sparse later-calendar subsets cannot confirm incremental skill.
  - 7-day Sharpe: 45 target-valid paired origins, 45 defined IC origins. Sparse later-calendar subsets cannot confirm incremental skill.
  - 30-day CAGR: 24 target-valid paired origins, 24 defined IC origins. Sparse later-calendar subsets cannot confirm incremental skill.
  - 30-day K-ratio: 24 target-valid paired origins, 24 defined IC origins. Sparse later-calendar subsets cannot confirm incremental skill.
  - 30-day Sharpe: 24 target-valid paired origins, 24 defined IC origins. Sparse later-calendar subsets cannot confirm incremental skill.
- **Existing missing=False:** positive K delta in 6/6 supported cells; observed delta range +0.002 to +0.016. This range is not a confidence bound.
- **7-day CAGR, cadence/delay:** 4/5 positive point estimates; range -0.005 to +0.003.
- **7-day K-ratio, cadence/delay:** 5/5 positive point estimates; range +0.004 to +0.010.
- **7-day Sharpe, cadence/delay:** 5/5 positive point estimates; range +0.005 to +0.013.
- **30-day CAGR, cadence/delay:** 5/5 positive point estimates; range +0.001 to +0.003.
- **30-day K-ratio, cadence/delay:** 5/5 positive point estimates; range +0.003 to +0.007.
- **30-day Sharpe, cadence/delay:** 5/5 positive point estimates; range +0.008 to +0.014.

## Baselines and calibration

- **7-day CAGR versus Constant:** same-row modelling-scale MAE change -0.5% (45 origins).
- **7-day K-ratio versus Constant:** same-row modelling-scale MAE change -1.1% (45 origins).
- **7-day Sharpe versus Constant:** same-row modelling-scale MAE change -0.7% (45 origins).
- **30-day CAGR versus Constant:** same-row modelling-scale MAE change -4.7% (24 origins).
- **30-day K-ratio versus Constant:** same-row modelling-scale MAE change -2.0% (24 origins).
- **30-day Sharpe versus Constant:** same-row modelling-scale MAE change -1.2% (24 origins).
- **7-day CAGR versus Zero return:** same-row modelling-scale MAE change +17.9% (45 origins).
- **30-day CAGR versus Zero return:** same-row modelling-scale MAE change +25.3% (24 origins).
- **7-day CAGR versus Persistence:** same-row modelling-scale MAE change -21.0% (45 origins).
- **7-day K-ratio versus Persistence:** same-row modelling-scale MAE change -22.0% (45 origins).
- **7-day Sharpe versus Persistence:** same-row modelling-scale MAE change -26.1% (45 origins).
- **30-day CAGR versus Persistence:** same-row modelling-scale MAE change -16.3% (24 origins).
- **30-day K-ratio versus Persistence:** same-row modelling-scale MAE change -21.2% (24 origins).
- **30-day Sharpe versus Persistence:** same-row modelling-scale MAE change -22.5% (24 origins).
- **7-day CAGR versus Quality/age:** same-row modelling-scale MAE change +5.2% (45 origins).
- **7-day K-ratio versus Quality/age:** same-row modelling-scale MAE change -0.9% (45 origins).
- **7-day Sharpe versus Quality/age:** same-row modelling-scale MAE change -0.5% (45 origins).
- **30-day CAGR versus Quality/age:** same-row modelling-scale MAE change +4.0% (24 origins).
- **30-day K-ratio versus Quality/age:** same-row modelling-scale MAE change -1.0% (24 origins).
- **30-day Sharpe versus Quality/age:** same-row modelling-scale MAE change +0.2% (24 origins).
- **7-day CAGR, original versus amended:** same-row MSE 1.676e+01 → 2.271e-01; IC 0.048 → 0.031.
- **7-day K-ratio, original versus amended:** same-row MSE 3.513e+01 → 1.119e+00; IC 0.127 → 0.145.
- **7-day Sharpe, original versus amended:** same-row MSE 2.038e+03 → 7.610e+01; IC 0.120 → 0.132.
- **30-day CAGR, original versus amended:** same-row MSE 8.534e+02 → 1.207e+00; IC 0.113 → 0.065.
- **30-day K-ratio, original versus amended:** same-row MSE 2.540e+01 → 1.053e+00; IC 0.177 → 0.171.
- **30-day Sharpe, original versus amended:** same-row MSE 7.720e+02 → 1.620e+01; IC 0.148 → 0.137.
- **7-day CAGR, amended prediction range:** [-4.810e-01, 1.376e-01]; output bound applied to 0.0% of forecasts.
- **7-day K-ratio, amended prediction range:** [-3.413e+00, 4.703e+00]; output bound applied to 0.0% of forecasts.
- **7-day Sharpe, amended prediction range:** [-4.626e+00, 5.541e+00]; output bound applied to 0.0% of forecasts.
- **30-day CAGR, amended prediction range:** [-1.248e+00, 5.468e-01]; output bound applied to 0.0% of forecasts.
- **30-day K-ratio, amended prediction range:** [-3.236e+00, 3.625e+00]; output bound applied to 0.0% of forecasts.
- **30-day Sharpe, amended prediction range:** [-2.775e+00, 4.039e+00]; output bound applied to 0.0% of forecasts.
- **Current-policy gate, 7-day CAGR:** paired K IC delta +0.004; 45 defined origins. Membership is caveated, not historically certified.
- **Current-policy gate, 7-day K-ratio:** paired K IC delta +0.017; 45 defined origins. Membership is caveated, not historically certified.
- **Current-policy gate, 7-day Sharpe:** paired K IC delta +0.018; 45 defined origins. Membership is caveated, not historically certified.
- **Current-policy gate, 30-day CAGR:** paired K IC delta +0.004; 24 defined origins. Membership is caveated, not historically certified.
- **Current-policy gate, 30-day K-ratio:** paired K IC delta +0.027; 24 defined origins. Membership is caveated, not historically certified.
- **Current-policy gate, 30-day Sharpe:** paired K IC delta +0.028; 24 defined origins. Membership is caveated, not historically certified.

## Matched lookback and sampling evidence

- **30d K-ratio → 30-day CAGR:** common-row IC 0.036; 62 defined origins and 15,618 rows, identical within this lookback comparison.
- **60d K-ratio → 30-day CAGR:** common-row IC 0.036; 62 defined origins and 15,618 rows, identical within this lookback comparison.
- **90d K-ratio → 30-day CAGR:** common-row IC 0.034; 62 defined origins and 15,618 rows, identical within this lookback comparison.
- **30d K-ratio → 30-day Sharpe:** common-row IC 0.085; 62 defined origins and 14,558 rows, identical within this lookback comparison.
- **60d K-ratio → 30-day Sharpe:** common-row IC 0.077; 62 defined origins and 14,558 rows, identical within this lookback comparison.
- **90d K-ratio → 30-day Sharpe:** common-row IC 0.077; 62 defined origins and 14,558 rows, identical within this lookback comparison.
- **30d K-ratio → 30-day K-ratio:** common-row IC 0.100; 62 defined origins and 14,764 rows, identical within this lookback comparison.
- **60d K-ratio → 30-day K-ratio:** common-row IC 0.098; 62 defined origins and 14,764 rows, identical within this lookback comparison.
- **90d K-ratio → 30-day K-ratio:** common-row IC 0.095; 62 defined origins and 14,764 rows, identical within this lookback comparison.
- **Snapshot conditioning:** 101 of 615 already archived addresses are outside the selected universe at 2026-10-01; this is a conditional study, not a complete historical-vault evaluation.

## Method and robustness

All continuous model inputs use training-only median-absolute asinh scales and explicit training-range/±5z bounds. CAGR inputs use exact lookback log returns. Sharpe fits a fixed scaled pseudo-Huber objective; K fits fold-scaled asinh targets. Level predictions use training 0.5%–99.5% target bounds before inverse transformation; labels and calculation tails remain untrimmed. Baselines are unbounded. Ranking companions have IC only, not level error.

Complete-case uses the fixed 30/60/90-day inventory, while partial history is measured over the supported ≤180-day inventory. Archive age is not inception. Missing=True is mechanical support for usable shorter inputs; same-row cohort errors separately assess usefulness. Zero-fill keeps shared indicators and is not a legacy selector replay.

The low-activity sensitivity requires max(4, ceil(window/2)) non-zero returns for Sharpe/K windows. Actual unchanged marks are valid primary observations. Discrete Sharpe/K stripes can reflect inactivity or precision; they do not prove stale prices. Activity sensitivity predictions use historical information only; realised future activity defines label validity afterwards, not forecast or entry eligibility. Calendar mark coverage and attrition are reported without inventing payouts.

Cadence phases, one-day delay, source-filled feature prices, quarterly stability, matched lookbacks/weekly sampling, calendar dependence, leave-vault/leave-block checks and explicit missing-outcome scenarios are exploratory. Bands do not eliminate dependence; the measured number of usable calendar blocks determines horizon support.

Current snapshot identities are not a certified historical listing. Zero assets/zero supply marks do not establish −100% investor loss. Inception, historical universe completeness and actual receipt/publication clocks remain unverified; no prospective collection pipeline is claimed. No strategy performance or deployment decision follows.

## Reproducibility

[Executed notebook](09-research-k-ratio-feature-forecasts.ipynb) · [Amendment](09-k-ratio-feature-forecast-amendment.md) · [Original implementation review](09-k-ratio-feature-forecast-implementation-review.md). Original code/notebook/headline outputs are preserved in the ignored `09-before-robust-amendment` namespace; current results and folds are under `09`.


### Helper source fingerprints

2026-01-01 → 2026-10-05 (end exclusive, UTC). Together with the pinned archive hash and model/data manifests, these identify the exact experiment implementation.

File,SHA-256
k_ratio_features.py,3de1fb81ab2bc27fb08fc2616eda1ac256ad3619eef794efe32918ec6d6230f0
k_ratio_forecast_audit.py,64895f0f4df76a959ac2caeb9e9d89e3d5a1c5f5479ccce434d783d88e83f3ca
k_ratio_forecast_extensions.py,6ce6ca65966630a20aaddf2cf21124c63faee4ae2c6c5d84765c3daab45f7571
k_ratio_forecast_feedback.py,f38684a14b696e77ca613a72dee8dcd84f9357e4692989a78646ff2f173ef5aa
k_ratio_forecast_models.py,dfe6b4446468872ecfa50b3ac0e3eb8bf961014b094a5c4b9ee0f8c847f7ebbf
k_ratio_forecast_report.py,448ee715a0ce2106a0bf4fab60fdfb6084cee12a6618adba16ce36c0a0dc7394
k_ratio_forecast_statistics.py,051ef7d8e9d5b0596cdb1408ffe2a8bc9f49219ff334ced9a7fdecab6fa285e7
